# 04 - Merge LoRA & Quantize (Colab)

For each trained LoRA rank (4, 8, 16): merges the adapter into the base model (`outputs/merged_r{rank}/`), then quantizes to 4-bit GPTQ (`outputs/quantized_r{rank}/`).

**Before running:** `Runtime > Change runtime type > T4 GPU` (or better). GPTQ quantization needs a real CUDA GPU. **Requires Phase 4's adapters** to already exist at `outputs/sft_r{rank}/final/` — if you trained with `USE_DRIVE = True` in `03_qlora_training.ipynb`, point `DRIVE_WORKDIR` below at the same Drive folder to pick them up.

In [ ]:
REPO_URL = "https://github.com/Shhaurya17/Efficient-Small-Language-Model-Adaptation-Quantization-Benchmark.git"  # public repo, no token needed
USE_DRIVE = True  # should match what you used in 03_qlora_training.ipynb, so adapters are found
DRIVE_WORKDIR = "/content/drive/MyDrive/efficient-slm-benchmark"

import os

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_WORKDIR, exist_ok=True)

REPO_DIR = os.path.join(DRIVE_WORKDIR, "repo") if USE_DRIVE else "/content/efficient-slm-benchmark"

if REPO_URL and not os.path.exists(os.path.join(REPO_DIR, ".git")):
    !git clone -q {REPO_URL} {REPO_DIR}

HAVE_REPO = os.path.exists(os.path.join(REPO_DIR, "configs", "quantize.yaml"))
OUTPUT_ROOT = os.path.join(REPO_DIR, "outputs") if HAVE_REPO else os.path.join(DRIVE_WORKDIR, "outputs")
print("Repo available:", HAVE_REPO)
print("Output root:", OUTPUT_ROOT)

for rank in (4, 8, 16):
    adapter_dir = os.path.join(OUTPUT_ROOT, f"sft_r{rank}", "final")
    print(rank, "adapter found:", os.path.exists(os.path.join(adapter_dir, "adapter_model.safetensors")))

In [ ]:
%%capture
!pip install -q transformers>=4.44.0 accelerate>=0.33.0 peft>=0.12.0 optimum>=1.21.0 auto-gptq>=0.7.1 pyyaml

In [ ]:
import torch

assert torch.cuda.is_available(), "GPTQ quantization requires a CUDA GPU. Set Runtime > Change runtime type > GPU."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import sys
import yaml

DEFAULT_MODEL_CONFIG = {"model_name": "Qwen/Qwen2.5-1.5B-Instruct", "torch_dtype": "float16"}
DEFAULT_QUANTIZE_CONFIG = {
    "quantization_method": "gptq",
    "gptq_config": {"bits": 4, "group_size": 128, "desc_act": False, "exllama": True, "version": 0},
    "calibration_dataset": "wikitext",
}

if HAVE_REPO:
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))
    with open(os.path.join(REPO_DIR, "configs", "model.yaml")) as f:
        model_config = yaml.safe_load(f)
    with open(os.path.join(REPO_DIR, "configs", "quantize.yaml")) as f:
        quantize_config = yaml.safe_load(f)
else:
    model_config, quantize_config = DEFAULT_MODEL_CONFIG, DEFAULT_QUANTIZE_CONFIG

try:
    from efficient_slm.quantization.merge import load_base_and_adapter, merge_lora, save_merged_model
    from efficient_slm.quantization.gptq import load_merged_model, quantize_model_gptq, save_quantized_model, verify_quantized_model
except ImportError as e:
    raise RuntimeError("Could not import efficient_slm. Set REPO_URL to clone the repo.") from e

print("Model:", model_config["model_name"])
print(quantize_config)

## Merge + quantize each rank

In [ ]:
import gc


def dir_size_gb(path):
    total = 0
    for root, _, files in os.walk(path):
        for name in files:
            total += os.path.getsize(os.path.join(root, name))
    return total / 1e9


RANKS_TO_RUN = [4, 8, 16]
size_comparison = {}

for rank in RANKS_TO_RUN:
    adapter_dir = os.path.join(OUTPUT_ROOT, f"sft_r{rank}", "final")
    merged_dir = os.path.join(OUTPUT_ROOT, f"merged_r{rank}")
    quantized_dir = os.path.join(OUTPUT_ROOT, f"quantized_r{rank}")

    if not os.path.exists(os.path.join(adapter_dir, "adapter_model.safetensors")):
        print(f"R={rank}: no adapter found at {adapter_dir}, skipping (run 03_qlora_training.ipynb first)")
        continue

    if not os.path.exists(os.path.join(merged_dir, "model.safetensors")):
        print(f"=== Merging R={rank} ===")
        peft_model, tokenizer = load_base_and_adapter(model_config["model_name"], adapter_dir, torch_dtype=model_config.get("torch_dtype", "float16"))
        merged_model = merge_lora(peft_model)
        save_merged_model(merged_model, tokenizer, merged_dir)
        del peft_model, merged_model
        gc.collect()
        torch.cuda.empty_cache()
    else:
        print(f"R={rank}: merged model already exists, skipping merge")

    if not os.path.exists(os.path.join(quantized_dir, "quantize_config.json")):
        print(f"=== Quantizing R={rank} (GPTQ, calibrating on {quantize_config['calibration_dataset']}) ===")
        model, tokenizer = load_merged_model(merged_dir, torch_dtype=model_config.get("torch_dtype", "float16"))
        quantized_model, quantizer = quantize_model_gptq(model, tokenizer, quantize_config)
        save_quantized_model(quantized_model, quantizer, tokenizer, quantized_dir)
        del model, quantized_model
        gc.collect()
        torch.cuda.empty_cache()
    else:
        print(f"R={rank}: quantized model already exists, skipping quantization")

    generation = verify_quantized_model(quantized_dir, prompt="Explain what a neural network is in one sentence.")
    print(f"R={rank} sanity generation: {generation}")

    size_comparison[f"r{rank}"] = {
        "merged_gb": dir_size_gb(merged_dir),
        "quantized_gb": dir_size_gb(quantized_dir),
    }
    print(f"R={rank} sizes: {size_comparison[f'r{rank}']}")

size_comparison

## Save size comparison table

In [ ]:
import json

with open(os.path.join(OUTPUT_ROOT, "size_comparison.json"), "w") as f:
    json.dump(size_comparison, f, indent=2)

print(json.dumps(size_comparison, indent=2))

## Getting models back to your local repo

`merged_r{rank}/` and `quantized_r{rank}/` are gitignored (large `*.safetensors`) — they stay in your Drive-backed `OUTPUT_ROOT`. Phase 6 (profiling) and Phase 7 (full evaluation) read directly from `outputs/quantized_r{rank}/` and `outputs/merged_r{rank}/`, so as long as those notebooks mount the same Drive folder, no manual copying is needed.